In [1]:
import os
import glob
import json
import time
import shutil
import numpy as np
import tensorflow as tf
from google.colab import drive

# Mount Drive
drive.mount("/content/drive")

# Define paths
BASE_PATH = "/content/drive/MyDrive/Translation Language Translator "
DRIVE_TOKENIZED_PATH = os.path.join(BASE_PATH, "tokenized")
DRIVE_MODEL_PATH = os.path.join(BASE_PATH, "models")
LOCAL_TOKENIZED_PATH = "/content/tokenized"
LOCAL_MODEL_PATH = "/content/models"

os.makedirs(LOCAL_TOKENIZED_PATH, exist_ok=True)
os.makedirs(LOCAL_MODEL_PATH, exist_ok=True)

print("TensorFlow version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
print("Paths configured.")

Mounted at /content/drive
TensorFlow version: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Paths configured.


In [2]:
drive_chunks = sorted(glob.glob(os.path.join(DRIVE_TOKENIZED_PATH, "train_chunk_*.npz")))
print(f"Training chunks found: {len(drive_chunks)}")

for src in drive_chunks:
    filename = os.path.basename(src)
    dst = os.path.join(LOCAL_TOKENIZED_PATH, filename)

    if not os.path.exists(dst):
        print("Copying:", filename)
        shutil.copy2(src, dst)
    else:
        print("Already exists:", filename)

Training chunks found: 14
Copying: train_chunk_000.npz
Copying: train_chunk_001.npz
Copying: train_chunk_002.npz
Copying: train_chunk_003.npz
Copying: train_chunk_004.npz
Copying: train_chunk_005.npz
Copying: train_chunk_006.npz
Copying: train_chunk_007.npz
Copying: train_chunk_008.npz
Copying: train_chunk_009.npz
Copying: train_chunk_010.npz
Copying: train_chunk_011.npz
Copying: train_chunk_012.npz
Copying: train_chunk_013.npz


In [12]:
import os
import json

CONFIG_PATH = os.path.join(DRIVE_MODEL_PATH, "transformer_config.json")
with open(CONFIG_PATH, "r") as f:
    config = json.load(f)

D_MODEL = config["d_model"]
NUM_HEADS = config["num_heads"]
NUM_ENCODER_LAYERS = config["num_encoder_layers"]
NUM_DECODER_LAYERS = config["num_decoder_layers"]
D_FF = config["d_ff"]
DROPOUT_RATE = config["dropout_rate"]
ENGLISH_VOCAB_SIZE = config["english_vocab_size"]
HINDI_VOCAB_SIZE = config["hindi_vocab_size"]
MAX_ENGLISH_LEN = config["max_english_len"]
MAX_HINDI_LEN = config["max_hindi_len"]
PAD_ID = config["pad_id"]
UNK_ID = config["unk_id"]
BOS_ID = config["bos_id"]
EOS_ID = config["eos_id"]

BATCH_SIZE = 512
EPOCHS = 20
CLIP_NORM = 1.0
WARMUP_STEPS = 3000

print(f"Model: d_model={D_MODEL}, heads={NUM_HEADS}, layers={NUM_ENCODER_LAYERS}/{NUM_DECODER_LAYERS}")
print(f"Vocab: EN={ENGLISH_VOCAB_SIZE}, HI={HINDI_VOCAB_SIZE}")
print(f"Batch: {BATCH_SIZE}, Epochs: {EPOCHS}")

Model: d_model=128, heads=4, layers=2/2
Vocab: EN=16000, HI=16000
Batch: 512, Epochs: 20


In [13]:
import os
import glob
import numpy as np

# 1. Re-define the paths (in case they were lost)
BASE_PATH = "/content/drive/MyDrive/Translation Language Translator "
TOKENIZED_PATH = f"{BASE_PATH}/tokenized"

# 2. Re-define chunk_files
chunk_files = sorted(glob.glob(f"{TOKENIZED_PATH}/train_chunk_*.npz"))
print("Number of chunks found:", len(chunk_files))
print("First 5 files:")
for f in chunk_files[:5]:
    print(f)

# 3. Now run the data verification
if len(chunk_files) > 0:
    data = np.load(chunk_files[0])
    print("\n--- DATA VERIFICATION ---")
    print("Encoder inputs shape:", data["encoder_inputs"].shape)
    print("Decoder inputs shape:", data["decoder_inputs"].shape)
    print("Decoder targets shape:", data["decoder_targets"].shape)

    print("\nUnique values in encoder_inputs (first 1000):", np.unique(data["encoder_inputs"][:1000])[:10])
    print("PAD_ID from config:", PAD_ID)
    print("Does encoder_inputs contain PAD_ID?", PAD_ID in np.unique(data["encoder_inputs"][:1000]))

    # Check if there are any empty sequences (all PAD)
    encoder_lengths = np.sum(data["encoder_inputs"] != PAD_ID, axis=1)
    print("\nMin encoder length:", np.min(encoder_lengths))
    print("Max encoder length:", np.max(encoder_lengths))

    decoder_lengths = np.sum(data["decoder_targets"] != PAD_ID, axis=1)
    print("Min decoder target length:", np.min(decoder_lengths))
    print("Max decoder target length:", np.max(decoder_lengths))

    # Check if loss is nan due to empty batches
    empty_sequences = np.sum(encoder_lengths == 0)
    print(f"\nNumber of completely empty encoder sequences: {empty_sequences}")
else:
    print("No chunk files found! Please check that your tokenized data is in Google Drive.")

Number of chunks found: 14
First 5 files:
/content/drive/MyDrive/Translation Language Translator /tokenized/train_chunk_000.npz
/content/drive/MyDrive/Translation Language Translator /tokenized/train_chunk_001.npz
/content/drive/MyDrive/Translation Language Translator /tokenized/train_chunk_002.npz
/content/drive/MyDrive/Translation Language Translator /tokenized/train_chunk_003.npz
/content/drive/MyDrive/Translation Language Translator /tokenized/train_chunk_004.npz

--- DATA VERIFICATION ---
Encoder inputs shape: (100000, 56)
Decoder inputs shape: (100000, 64)
Decoder targets shape: (100000, 64)

Unique values in encoder_inputs (first 1000): [ 0  4  6  7  8 10 11 13 14 15]
PAD_ID from config: 0
Does encoder_inputs contain PAD_ID? True

Min encoder length: 1
Max encoder length: 56
Min decoder target length: 2
Max decoder target length: 64

Number of completely empty encoder sequences: 0


In [14]:
# ==========================================
# 1. Positional Encoding
# ==========================================
def positional_encoding(max_length, d_model):
    positions = np.arange(max_length)[:, np.newaxis]
    dimensions = np.arange(d_model)[np.newaxis, :]
    angle_rates = 1 / np.power(10000, (2 * (dimensions // 2)) / np.float32(d_model))
    angle_radians = positions * angle_rates
    encoding = np.zeros((max_length, d_model))
    encoding[:, 0::2] = np.sin(angle_radians[:, 0::2])
    encoding[:, 1::2] = np.cos(angle_radians[:, 1::2])
    return tf.cast(encoding, dtype=tf.float32)

# ==========================================
# 2. Token Embedding
# ==========================================
class TokenEmbedding(tf.keras.layers.Layer):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.d_model = d_model
        self.embedding = tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=d_model)

    def call(self, x):
        x = self.embedding(x)
        scale = tf.math.sqrt(tf.cast(self.d_model, dtype=x.dtype))
        return x * scale

# ==========================================
# 3. Scaled Dot-Product Attention
# ==========================================
def scaled_dot_product_attention(query, key, value, mask=None):
    scores = tf.matmul(query, key, transpose_b=True)
    depth = tf.cast(tf.shape(key)[-1], scores.dtype)
    scores = scores / tf.math.sqrt(depth)
    if mask is not None:
        mask = tf.cast(mask, scores.dtype)
        scores += (mask * tf.cast(-1e4, scores.dtype))
    scores = tf.cast(scores, tf.float32)
    attention_weights = tf.nn.softmax(scores, axis=-1)
    attention_weights = tf.cast(attention_weights, value.dtype)
    output = tf.matmul(attention_weights, value)
    return output, attention_weights

# ==========================================
# 4. Feed Forward Network
# ==========================================
def feed_forward_network(d_model, d_ff):
    return tf.keras.Sequential([
        tf.keras.layers.Dense(d_ff, activation="relu"),
        tf.keras.layers.Dense(d_model)
    ])

# ==========================================
# 5. Multi-Head Attention
# ==========================================
class MultiHeadAttention(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.supports_masking = False
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.depth = d_model // num_heads
        self.wq = tf.keras.layers.Dense(d_model)
        self.wk = tf.keras.layers.Dense(d_model)
        self.wv = tf.keras.layers.Dense(d_model)
        self.dense = tf.keras.layers.Dense(d_model)

    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, query, key, value, mask=None, **kwargs):
        batch_size = tf.shape(query)[0]
        query = self.wq(query)
        key = self.wk(key)
        value = self.wv(value)
        query = self.split_heads(query, batch_size)
        key = self.split_heads(key, batch_size)
        value = self.split_heads(value, batch_size)
        attention_output, attention_weights = scaled_dot_product_attention(query, key, value, mask)
        attention_output = tf.transpose(attention_output, perm=[0, 2, 1, 3])
        attention_output = tf.reshape(attention_output, (batch_size, -1, self.d_model))
        output = self.dense(attention_output)
        return output, attention_weights

# ==========================================
# 6. Encoder Layer
# ==========================================
class EncoderLayer(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, d_ff, dropout_rate):
        super().__init__()
        self.supports_masking = False
        self.self_attention = MultiHeadAttention(d_model, num_heads)
        self.ffn = feed_forward_network(d_model, d_ff)
        self.norm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = tf.keras.layers.Dropout(dropout_rate)
        self.dropout2 = tf.keras.layers.Dropout(dropout_rate)

    def call(self, x, training=False, mask=None, **kwargs):
        attention_output, _ = self.self_attention(x, x, x, mask)
        attention_output = self.dropout1(attention_output, training=training)
        out1 = self.norm1(x + attention_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        out2 = self.norm2(out1 + ffn_output)
        return out2

# ==========================================
# 7. Encoder
# ==========================================
class Encoder(tf.keras.layers.Layer):
    def __init__(self, num_layers, d_model, num_heads, d_ff, vocab_size, max_length, dropout_rate=0.1):
        super().__init__()
        self.supports_masking = False
        self.d_model = d_model
        self.num_layers = num_layers
        self.embedding = TokenEmbedding(vocab_size, d_model)
        self.pos_encoding = positional_encoding(max_length, d_model)
        self.enc_layers = [EncoderLayer(d_model, num_heads, d_ff, dropout_rate) for _ in range(num_layers)]
        self.dropout = tf.keras.layers.Dropout(dropout_rate)

    def call(self, x, training=False, mask=None, **kwargs):
        seq_length = tf.shape(x)[1]
        x = self.embedding(x)
        pos_encoding = tf.cast(self.pos_encoding[:seq_length], x.dtype)
        x = x + pos_encoding
        x = self.dropout(x, training=training)
        for i in range(self.num_layers):
            x = self.enc_layers[i](x, training=training, mask=mask)
        return x

# ==========================================
# 8. Decoder Layer
# ==========================================
class DecoderLayer(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, d_ff, dropout_rate):
        super().__init__()
        self.supports_masking = False
        self.self_attention = MultiHeadAttention(d_model, num_heads)
        self.cross_attention = MultiHeadAttention(d_model, num_heads)
        self.ffn = feed_forward_network(d_model, d_ff)
        self.norm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.norm3 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = tf.keras.layers.Dropout(dropout_rate)
        self.dropout2 = tf.keras.layers.Dropout(dropout_rate)
        self.dropout3 = tf.keras.layers.Dropout(dropout_rate)

    def call(self, x, encoder_output, training=False, look_ahead_mask=None, padding_mask=None, **kwargs):
        attn1, attn_weights_block1 = self.self_attention(x, x, x, look_ahead_mask)
        attn1 = self.dropout1(attn1, training=training)
        out1 = self.norm1(x + attn1)
        attn2, attn_weights_block2 = self.cross_attention(out1, encoder_output, encoder_output, padding_mask)
        attn2 = self.dropout2(attn2, training=training)
        out2 = self.norm2(out1 + attn2)
        ffn_output = self.ffn(out2)
        ffn_output = self.dropout3(ffn_output, training=training)
        out3 = self.norm3(out2 + ffn_output)
        return out3, attn_weights_block1, attn_weights_block2

# ==========================================
# 9. Decoder
# ==========================================
class Decoder(tf.keras.layers.Layer):
    def __init__(self, num_layers, d_model, num_heads, d_ff, vocab_size, max_length, dropout_rate=0.1):
        super().__init__()
        self.supports_masking = False
        self.d_model = d_model
        self.num_layers = num_layers
        self.embedding = TokenEmbedding(vocab_size, d_model)
        self.pos_encoding = positional_encoding(max_length, d_model)
        self.dec_layers = [DecoderLayer(d_model, num_heads, d_ff, dropout_rate) for _ in range(num_layers)]
        self.dropout = tf.keras.layers.Dropout(dropout_rate)

    def call(self, x, encoder_output, training=False, look_ahead_mask=None, padding_mask=None, **kwargs):
        seq_length = tf.shape(x)[1]
        x = self.embedding(x)
        pos_encoding = tf.cast(self.pos_encoding[:seq_length], x.dtype)
        x = x + pos_encoding
        x = self.dropout(x, training=training)
        attention_weights = {}
        for i in range(self.num_layers):
            x, block1, block2 = self.dec_layers[i](
                x, encoder_output, training=training,
                look_ahead_mask=look_ahead_mask, padding_mask=padding_mask
            )
            attention_weights[f"decoder_layer{i+1}_block1"] = block1
            attention_weights[f"decoder_layer{i+1}_block2"] = block2
        return x, attention_weights

# ==========================================
# 10. Transformer
# ==========================================
class Transformer(tf.keras.Model):
    def __init__(self, num_encoder_layers, num_decoder_layers, d_model, num_heads, d_ff,
                 input_vocab_size, target_vocab_size, input_max_length, target_max_length, dropout_rate):
        super().__init__()
        self.encoder = Encoder(num_encoder_layers, d_model, num_heads, d_ff, input_vocab_size, input_max_length, dropout_rate)
        self.decoder = Decoder(num_decoder_layers, d_model, num_heads, d_ff, target_vocab_size, target_max_length, dropout_rate)
        self.final_layer = tf.keras.layers.Dense(target_vocab_size)

    def call(self, inp, tar, training=False, encoder_padding_mask=None, decoder_look_ahead_mask=None, decoder_padding_mask=None, **kwargs):
        enc_output = self.encoder(inp, training=training, mask=encoder_padding_mask)
        dec_output, attention_weights = self.decoder(
            tar, enc_output, training=training,
            look_ahead_mask=decoder_look_ahead_mask, padding_mask=decoder_padding_mask
        )
        final_output = self.final_layer(dec_output)
        return final_output, attention_weights, enc_output, dec_output

In [15]:
def create_padding_mask(sequence):
    mask = tf.cast(tf.equal(sequence, PAD_ID), tf.float32)
    return mask[:, tf.newaxis, tf.newaxis, :]

def create_look_ahead_mask(size):
    mask = 1 - tf.linalg.band_part(tf.ones((size, size)), -1, 0)
    return mask

def create_masks(encoder_inputs, decoder_inputs):
    encoder_padding_mask = create_padding_mask(encoder_inputs)
    decoder_padding_mask = create_padding_mask(encoder_inputs)
    decoder_target_padding_mask = create_padding_mask(decoder_inputs)
    target_length = tf.shape(decoder_inputs)[1]
    look_ahead_mask = create_look_ahead_mask(target_length)
    combined_decoder_mask = tf.maximum(
        decoder_target_padding_mask,
        look_ahead_mask[tf.newaxis, tf.newaxis, :, :]
    )
    return encoder_padding_mask, combined_decoder_mask, decoder_padding_mask

def load_chunk_dataset(chunk_file, batch_size):
    data = np.load(chunk_file, allow_pickle=False)
    encoder_inputs = data["encoder_inputs"]
    decoder_inputs = data["decoder_inputs"]
    decoder_targets = data["decoder_targets"]
    dataset = tf.data.Dataset.from_tensor_slices((encoder_inputs, decoder_inputs, decoder_targets))
    dataset = dataset.shuffle(min(len(encoder_inputs), 10000), reshuffle_each_iteration=True)
    dataset = dataset.batch(batch_size, drop_remainder=True)
    dataset = dataset.cache()
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

In [16]:
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction="none")

def masked_loss(real, predicted):
    loss = loss_object(real, predicted)
    mask = tf.cast(tf.not_equal(real, PAD_ID), loss.dtype)
    loss *= mask
    return tf.reduce_sum(loss) / (tf.reduce_sum(mask) + 1e-9)

def masked_accuracy(real, predicted):
    predictions = tf.argmax(predicted, axis=-1)
    predictions = tf.cast(predictions, real.dtype)
    matches = tf.equal(real, predictions)
    mask = tf.not_equal(real, PAD_ID)
    matches = tf.logical_and(matches, mask)
    matches = tf.cast(matches, tf.float32)
    mask = tf.cast(mask, tf.float32)
    return tf.reduce_sum(matches) / (tf.reduce_sum(mask) + 1e-9)

class TransformerSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, d_model, warmup_steps=4000):
        super().__init__()
        self.d_model = tf.cast(d_model, tf.float32)
        self.warmup_steps = tf.cast(warmup_steps, tf.float32)
    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        arg1 = tf.math.rsqrt(step)
        arg2 = step * tf.math.pow(self.warmup_steps, -1.5)
        return tf.math.rsqrt(self.d_model) * tf.math.minimum(arg1, arg2)

learning_rate = TransformerSchedule(d_model=D_MODEL, warmup_steps=WARMUP_STEPS)
base_optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate, beta_1=0.9, beta_2=0.98, epsilon=1e-9)
optimizer = tf.keras.mixed_precision.LossScaleOptimizer(base_optimizer)
print("Optimizer created.")

Optimizer created.


In [17]:
# Instantiate
transformer = Transformer(
    num_encoder_layers=NUM_ENCODER_LAYERS,
    num_decoder_layers=NUM_DECODER_LAYERS,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    d_ff=D_FF,
    input_vocab_size=ENGLISH_VOCAB_SIZE,
    target_vocab_size=HINDI_VOCAB_SIZE,
    input_max_length=MAX_ENGLISH_LEN,
    target_max_length=MAX_HINDI_LEN,
    dropout_rate=DROPOUT_RATE
)

# ✅ FORCE BUILD — this creates the weights!
dummy_enc = tf.zeros((2, MAX_ENGLISH_LEN), dtype=tf.int32)
dummy_dec = tf.zeros((2, MAX_HINDI_LEN), dtype=tf.int32)
dummy_enc_mask, dummy_dec_look, dummy_dec_pad = create_masks(dummy_enc, dummy_dec)

_ = transformer(
    dummy_enc, dummy_dec,
    training=False,
    encoder_padding_mask=dummy_enc_mask,
    decoder_look_ahead_mask=dummy_dec_look,
    decoder_padding_mask=dummy_dec_pad
)

# ✅ VERIFY
n_vars = len(transformer.trainable_variables)
print(f"\n✅ Trainable variables: {n_vars}")
assert n_vars > 0, "❌ MODEL STILL HAS NO WEIGHTS!"
print("✅ Model is properly built and ready for training.")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'encoder_layer_2' (of type EncoderLayer) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'encoder_layer_3' (of type EncoderLayer) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'encoder_1' (of type Encoder) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(



✅ Trainable variables: 88
✅ Model is properly built and ready for training.


In [9]:
@tf.function(jit_compile=True)
def train_step(encoder_inputs, decoder_inputs, decoder_targets):
    with tf.GradientTape() as tape:
        (encoder_padding_mask, decoder_look_ahead_mask, decoder_padding_mask) = create_masks(
            encoder_inputs, decoder_inputs
        )
        predictions, _, _, _ = transformer(
            encoder_inputs, decoder_inputs,
            training=True,
            encoder_padding_mask=encoder_padding_mask,
            decoder_look_ahead_mask=decoder_look_ahead_mask,
            decoder_padding_mask=decoder_padding_mask
        )
        # No casting needed — everything is float32 now
        loss = masked_loss(decoder_targets, predictions)

    gradients = tape.gradient(loss, transformer.trainable_variables)

    # ✅ Debug: verify gradients are not None
    # (only prints during tracing, harmless)
    optimizer.apply_gradients(zip(gradients, transformer.trainable_variables))

    accuracy = masked_accuracy(decoder_targets, predictions)
    return loss, accuracy

In [10]:
# Check the first chunk for PAD_ID consistency
data = np.load(chunk_files[0])
print("Encoder inputs shape:", data["encoder_inputs"].shape)
print("Decoder inputs shape:", data["decoder_inputs"].shape)
print("Decoder targets shape:", data["decoder_targets"].shape)

print("Unique values in encoder_inputs (first 1000):", np.unique(data["encoder_inputs"][:1000])[:10])
print("PAD_ID from config:", PAD_ID)
print("Does encoder_inputs contain PAD_ID?", PAD_ID in np.unique(data["encoder_inputs"][:1000]))

# Check if there are any empty sequences (all PAD)
encoder_lengths = np.sum(data["encoder_inputs"] != PAD_ID, axis=1)
print("Min encoder length:", np.min(encoder_lengths))
print("Max encoder length:", np.max(encoder_lengths))

decoder_lengths = np.sum(data["decoder_targets"] != PAD_ID, axis=1)
print("Min decoder target length:", np.min(decoder_lengths))
print("Max decoder target length:", np.max(decoder_lengths))

Encoder inputs shape: (100000, 56)
Decoder inputs shape: (100000, 64)
Decoder targets shape: (100000, 64)
Unique values in encoder_inputs (first 1000): [ 0  4  6  7  8 10 11 13 14 15]
PAD_ID from config: 0
Does encoder_inputs contain PAD_ID? True
Min encoder length: 1
Max encoder length: 56
Min decoder target length: 2
Max decoder target length: 64


In [ ]:
#setup chechpoints and run training loop
# Checkpoint Setup
CHECKPOINT_PATH = os.path.join(DRIVE_MODEL_PATH, "training_v2")
os.makedirs(CHECKPOINT_PATH, exist_ok=True)

checkpoint = tf.train.Checkpoint(transformer=transformer, optimizer=optimizer)
checkpoint_manager = tf.train.CheckpointManager(checkpoint, directory=CHECKPOINT_PATH, max_to_keep=3)

# History tracking
history = {"train_loss": [], "train_accuracy": [], "epoch_time": []}
HISTORY_PATH = os.path.join(CHECKPOINT_PATH, "training_history.json")

# ==========================================
# TRAINING LOOP (FIXED)
# ==========================================
print(f"Starting training for {EPOCHS} epochs...")

for epoch in range(EPOCHS):  # FIX: range(EPOCHS) runs 0 to EPOCHS-1
    print(f"\n========== EPOCH {epoch + 1}/{EPOCHS} ==========")
    epoch_start = time.time()

    total_loss = 0.0
    total_accuracy = 0.0
    total_batches = 0

    for chunk_index, chunk_file in enumerate(chunk_files):
        print(f"\nChunk {chunk_index + 1}/{len(chunk_files)}")

        # Load dataset for this chunk
        train_dataset = load_chunk_dataset(chunk_file, BATCH_SIZE)

        chunk_loss = 0.0
        chunk_batches = 0

        for encoder_batch, decoder_batch, target_batch in train_dataset:
            loss, accuracy = train_step(encoder_batch, decoder_batch, target_batch)

            loss_value = loss.numpy()
            accuracy_value = accuracy.numpy()

            chunk_loss += loss_value
            total_loss += loss_value
            total_accuracy += accuracy_value

            chunk_batches += 1
            total_batches += 1

        print(f"Chunk loss: {chunk_loss / chunk_batches:.4f}")

    # Calculate epoch metrics
    mean_loss = total_loss / total_batches
    mean_accuracy = total_accuracy / total_batches
    epoch_time = time.time() - epoch_start

    # Save history
    history["train_loss"].append(float(mean_loss))
    history["train_accuracy"].append(float(mean_accuracy))
    history["epoch_time"].append(float(epoch_time))

    # Save Checkpoint
    checkpoint_path = checkpoint_manager.save()

    # Save History to Drive
    with open(HISTORY_PATH, "w") as f:
        json.dump(history, f, indent=4)

    # Print epoch summary
    print(f"\nEpoch {epoch + 1} completed")
    print(f"Loss     : {mean_loss:.4f}")
    print(f"Accuracy : {mean_accuracy:.4f}")
    print(f"Time     : {epoch_time / 60:.2f} minutes")
    print("Checkpoint saved:", checkpoint_path)

print("\nTraining Complete!")

Starting training for 10 epochs...

========== EPOCH 1/10 ==========

Chunk 1/14
Chunk loss: 9.2284

Chunk 2/14
Chunk loss: 7.5486

Chunk 3/14
Chunk loss: 6.6805

Chunk 4/14
Chunk loss: 6.5205

Chunk 5/14
Chunk loss: 5.9671

Chunk 6/14
Chunk loss: 5.3552

Chunk 7/14
Chunk loss: 5.6125

Chunk 8/14
Chunk loss: 5.3427

Chunk 9/14
Chunk loss: 4.9018

Chunk 10/14
Chunk loss: 4.8630

Chunk 11/14
Chunk loss: 5.0279

Chunk 12/14
Chunk loss: 4.7496

Chunk 13/14
Chunk loss: 3.9558

Chunk 14/14
Chunk loss: 4.0247

Epoch 1 completed
Loss     : 5.7297
Accuracy : 0.2038
Time     : 14.54 minutes
Checkpoint saved: /content/drive/MyDrive/Translation Language Translator /models/training_v2/ckpt-1

========== EPOCH 2/10 ==========

Chunk 1/14
Chunk loss: 3.6615

Chunk 2/14
Chunk loss: 3.7069

Chunk 3/14
Chunk loss: 3.3275

Chunk 4/14
Chunk loss: 4.2698

Chunk 5/14
Chunk loss: 3.9411

Chunk 6/14
Chunk loss: 4.2856

Chunk 7/14
Chunk loss: 4.2154

Chunk 8/14
Chunk loss: 3.8222

Chunk 9/14
Chunk loss: 3.3685

KeyboardInterrupt: 

In [11]:
# ==========================================
# RESTORE FROM CHECKPOINT & SKIP COMPLETED EPOCHS
# ==========================================
import json

CHECKPOINT_PATH = os.path.join(DRIVE_MODEL_PATH, "training_v2")

# 1. Restore model weights only (avoid optimizer dtype mismatch)
latest = tf.train.latest_checkpoint(CHECKPOINT_PATH)
print(f"Latest checkpoint: {latest}")

if latest is not None:
    # Model-only restore
    restore_ckpt = tf.train.Checkpoint(transformer=transformer)
    restore_ckpt.restore(latest).expect_partial()
    print(" Model weights restored.")
else:
    print(" No checkpoint found. Training will start from scratch.")

# 2. Load history to find how many epochs are done
HISTORY_PATH = os.path.join(CHECKPOINT_PATH, "training_history.json")
if os.path.exists(HISTORY_PATH):
    with open(HISTORY_PATH) as f:
        history = json.load(f)
    START_EPOCH = len(history["train_loss"])
    print(f" History loaded. Epochs completed: {START_EPOCH}")
    for i, (loss, acc) in enumerate(zip(history["train_loss"], history["train_accuracy"])):
        print(f"   Epoch {i+1}: loss={loss:.4f}, acc={acc*100:.2f}%")
else:
    history = {"train_loss": [], "train_accuracy": [], "epoch_time": []}
    START_EPOCH = 0
    print(" No history. Starting fresh.")

print(f"\n Will resume from Epoch {START_EPOCH + 1}")

Latest checkpoint: /content/drive/MyDrive/Translation Language Translator /models/training_v2/ckpt-6
 Model weights restored.
 History loaded. Epochs completed: 6
   Epoch 1: loss=5.7297, acc=20.38%
   Epoch 2: loss=3.7170, acc=36.43%
   Epoch 3: loss=3.2369, acc=41.42%
   Epoch 4: loss=3.0739, acc=43.15%
   Epoch 5: loss=3.0041, acc=43.86%
   Epoch 6: loss=2.9714, acc=44.18%

 Will resume from Epoch 7


In [20]:
# ==========================================
# ✅ RESTORE FROM CHECKPOINT — STRICT VERSION
# ==========================================
import json

CHECKPOINT_PATH = os.path.join(DRIVE_MODEL_PATH, "training_v2")

# 1. Find the checkpoint we want to restore from
#    Use ckpt-6 explicitly if it exists, else latest
TARGET_CKPT = os.path.join(CHECKPOINT_PATH, "ckpt-6")
if not os.path.exists(TARGET_CKPT + ".index"):
    print(f"⚠️  ckpt-6 not found, using latest available")
    TARGET_CKPT = tf.train.latest_checkpoint(CHECKPOINT_PATH)

print(f"📂 Restoring from: {TARGET_CKPT}")

# 2. Snapshot weights BEFORE restore
w_before = transformer.trainable_variables[0].numpy().copy()

# 3. Restore (strict — no expect_partial)
restore_ckpt = tf.train.Checkpoint(transformer=transformer)
status = restore_ckpt.restore(TARGET_CKPT)
status.assert_existing_objects_matched()
print("✅ Restore status: OK")

# 4. Snapshot weights AFTER restore
w_after = transformer.trainable_variables[0].numpy()
change = np.abs(w_before - w_after).sum()
print(f"📊 Weight change after restore: {change:.6f}")

if change < 1e-6:
    print("🚨 WARNING: Weights did NOT change! Restore is silently failing.")
    print("    This means the model architecture doesn't match the checkpoint.")
else:
    print("✅ Weights successfully restored from checkpoint!")

# 5. Load history
HISTORY_PATH = os.path.join(CHECKPOINT_PATH, "training_history.json")
if os.path.exists(HISTORY_PATH):
    with open(HISTORY_PATH) as f:
        history = json.load(f)
    START_EPOCH = len(history["train_loss"])
    print(f"\n📚 History loaded. Epochs completed: {START_EPOCH}")
    for i, (loss, acc) in enumerate(zip(history["train_loss"], history["train_accuracy"])):
        print(f"   Epoch {i+1}: loss={loss:.4f}, acc={acc*100:.2f}%")
else:
    history = {"train_loss": [], "train_accuracy": [], "epoch_time": []}
    START_EPOCH = 0

print(f"\n🚀 Will resume from Epoch {START_EPOCH + 1}")

📂 Restoring from: /content/drive/MyDrive/Translation Language Translator /models/training_v2/ckpt-6
✅ Restore status: OK
📊 Weight change after restore: 113868.062500
✅ Weights successfully restored from checkpoint!

📚 History loaded. Epochs completed: 7
   Epoch 1: loss=5.7297, acc=20.38%
   Epoch 2: loss=3.7170, acc=36.43%
   Epoch 3: loss=3.2369, acc=41.42%
   Epoch 4: loss=3.0739, acc=43.15%
   Epoch 5: loss=3.0041, acc=43.86%
   Epoch 6: loss=2.9714, acc=44.18%
   Epoch 7: loss=5.7212, acc=20.47%

🚀 Will resume from Epoch 8


In [25]:
import json
CKPT_DIR = "/content/drive/MyDrive/Translation Language Translator /models/training_v2"
HISTORY_PATH = f"{CKPT_DIR}/training_history.json"

with open(HISTORY_PATH) as f:
    history = json.load(f)

# Remove the last (bogus) epoch
print(f"Before: {len(history['train_loss'])} epochs")
history["train_loss"] = history["train_loss"][:6]
history["train_accuracy"] = history["train_accuracy"][:6]
history["epoch_time"] = history["epoch_time"][:6]
print(f"After: {len(history['train_loss'])} epochs")

with open(HISTORY_PATH, "w") as f:
    json.dump(history, f, indent=4)

print("✅ Bogus Epoch 7 removed from history")

Before: 7 epochs
After: 6 epochs
✅ Bogus Epoch 7 removed from history


In [26]:
import os
CKPT_DIR = "/content/drive/MyDrive/Translation Language Translator /models/training_v2"

# Delete the corrupted ckpt-1
for f in ["ckpt-1.data-00000-of-00001", "ckpt-1.index"]:
    path = os.path.join(CKPT_DIR, f)
    if os.path.exists(path):
        os.remove(path)
        print(f"🗑️  Deleted {f}")

🗑️  Deleted ckpt-1.data-00000-of-00001
🗑️  Deleted ckpt-1.index


In [31]:
import os
import numpy as np

CKPT_DIR = "/content/drive/MyDrive/Translation Language Translator /models/training_v2"

# 1. Create checkpoint object with a custom counter variable (named differently)
counter_var = tf.Variable(6, dtype=tf.int64, trainable=False, name="manager_step")
checkpoint = tf.train.Checkpoint(transformer=transformer, manager_step=counter_var)

# 2. Restore model weights from ckpt-6 (the good one)
restore_status = checkpoint.restore(os.path.join(CKPT_DIR, "ckpt-6"))
restore_status.expect_partial()

# 3. Verify restore
w = transformer.trainable_variables[0].numpy()
print(f"First 3 weights: {w.flatten()[:3]}")
print(f"Weight norm: {np.linalg.norm(w):.4f}")
print(f"Counter value: {counter_var.numpy()}")

# 4. Create manager with our custom counter
checkpoint_manager = tf.train.CheckpointManager(
    checkpoint,
    directory=CKPT_DIR,
    max_to_keep=5,
    checkpoint_name="ckpt",
    step_counter=counter_var,   # ✅ pass our custom counter
)

# 5. Force a save so the pointer file is rewritten to point at the right ckpt
save_path = checkpoint_manager.save()
print(f"\n✅ Saved new checkpoint: {save_path}")

# 6. Show pointer file
with open(os.path.join(CKPT_DIR, "checkpoint")) as f:
    print("\nNew pointer file:")
    print(f.read())

# 7. Final verification — print all ckpt files
print("\nAll files in checkpoint dir:")
for f in sorted(os.listdir(CKPT_DIR)):
    print(f"  {f}")

First 3 weights: [-0.04804884 -0.03730192 -0.02122419]
Weight norm: 82.1300
Counter value: 6

✅ Saved new checkpoint: /content/drive/MyDrive/Translation Language Translator /models/training_v2/ckpt-7

New pointer file:
model_checkpoint_path: "ckpt-7"
all_model_checkpoint_paths: "ckpt-5"
all_model_checkpoint_paths: "ckpt-6"
all_model_checkpoint_paths: "ckpt-1"
all_model_checkpoint_paths: "ckpt-7"
all_model_checkpoint_timestamps: 1790930913.915971
all_model_checkpoint_timestamps: 1790931699.3677843
all_model_checkpoint_timestamps: 1791004240.654112
all_model_checkpoint_timestamps: 1791004909.813487
last_preserved_timestamp: 1790926841.8797588


All files in checkpoint dir:
  checkpoint
  ckpt-5.data-00000-of-00001
  ckpt-5.index
  ckpt-6.data-00000-of-00001
  ckpt-6.index
  ckpt-7.data-00000-of-00001
  ckpt-7.index
  training_history.json


In [ ]:
# ==========================================
# TRAINING LOOP — FINAL WORKING VERSION
# ==========================================
import json

CHECKPOINT_PATH = os.path.join(DRIVE_MODEL_PATH, "training_v2")
HISTORY_PATH = os.path.join(CHECKPOINT_PATH, "training_history.json")

# Load current history (should now have 6 epochs)
with open(HISTORY_PATH) as f:
    history = json.load(f)
START_EPOCH = len(history["train_loss"])

print(f"🚀 Resuming from Epoch {START_EPOCH + 1} to {EPOCHS}")

# Checkpoint manager ALREADY exists (created in Step 3 above)
# Just use it — DO NOT recreate it

for epoch in range(START_EPOCH, EPOCHS):
    print(f"\n========== EPOCH {epoch + 1}/{EPOCHS} ==========")
    epoch_start = time.time()
    total_loss = 0.0
    total_accuracy = 0.0
    total_batches = 0

    for chunk_index, chunk_file in enumerate(chunk_files):
        print(f"\nChunk {chunk_index + 1}/{len(chunk_files)}")
        train_dataset = load_chunk_dataset(chunk_file, BATCH_SIZE)
        chunk_loss = 0.0
        chunk_batches = 0

        for encoder_batch, decoder_batch, target_batch in train_dataset:
            loss, accuracy = train_step(encoder_batch, decoder_batch, target_batch)
            loss_value = loss.numpy()
            accuracy_value = accuracy.numpy()
            chunk_loss += loss_value
            total_loss += loss_value
            total_accuracy += accuracy_value
            chunk_batches += 1
            total_batches += 1

        print(f"Chunk loss: {chunk_loss / chunk_batches:.4f}")

    mean_loss = total_loss / total_batches
    mean_accuracy = total_accuracy / total_batches
    epoch_time = time.time() - epoch_start

    history["train_loss"].append(float(mean_loss))
    history["train_accuracy"].append(float(mean_accuracy))
    history["epoch_time"].append(float(epoch_time))

    checkpoint_manager.save()

    with open(HISTORY_PATH, "w") as f:
        json.dump(history, f, indent=4)

    print(f"\nEpoch {epoch + 1} completed")
    print(f"Loss     : {mean_loss:.4f}")
    print(f"Accuracy : {mean_accuracy:.4f}")
    print(f"Time     : {epoch_time / 60:.2f} minutes")

print("\n🎉 Training Complete!")

🚀 Resuming from Epoch 7 to 20

========== EPOCH 7/20 ==========

Chunk 1/14
Chunk loss: 3.4035

Chunk 2/14
Chunk loss: 3.1276

Chunk 3/14
Chunk loss: 3.0158

Chunk 4/14
Chunk loss: 4.0057

Chunk 5/14
Chunk loss: 3.7323

Chunk 6/14
Chunk loss: 3.7761

Chunk 7/14
Chunk loss: 3.6017

Chunk 8/14
Chunk loss: 3.5069

Chunk 9/14
Chunk loss: 3.2022

Chunk 10/14
Chunk loss: 3.2289

Chunk 11/14
Chunk loss: 3.7673

Chunk 12/14
Chunk loss: 3.7818

Chunk 13/14
Chunk loss: 2.8926

Chunk 14/14
Chunk loss: 3.1069

Epoch 7 completed
Loss     : 3.4454
Accuracy : 0.3806
Time     : 13.66 minutes

========== EPOCH 8/20 ==========

Chunk 1/14
Chunk loss: 2.4575

Chunk 2/14
Chunk loss: 2.7740

Chunk 3/14
Chunk loss: 2.7479

Chunk 4/14
Chunk loss: 3.4853

Chunk 5/14
Chunk loss: 3.3521

Chunk 6/14
Chunk loss: 3.6230

Chunk 7/14
Chunk loss: 3.3395

Chunk 8/14
Chunk loss: 3.1560

Chunk 9/14
Chunk loss: 2.8650

Chunk 10/14
Chunk loss: 2.8700

Chunk 11/14
Chunk loss: 3.4972

Chunk 12/14
Chunk loss: 3.5905

Chunk 1